This version deals with five datasets: crc01 (scRNA-seq, HRA003293), crc03 (scRNA-seq, GSE234804), crc06 (snRNA-seq, GSE183916), crc09 (Pelka 2021), crc10 (Lee 2020)

In [1]:
### Import modules
import os
from pathlib import Path
import numpy as np
import pandas as pd
import scanpy as sc
import scipy as sp
import anndata as ad
from anndata import AnnData
import sys
import re
### Import personal Utensils
sys.path.insert(0, "..")
import utils as ut

In [2]:
### Set Data Dir
DataRootDir = Path("../../data")

### Data Curation

Doctrine:

A. raw data will be stored in .layers\['counts']

B. Unification of .obs metadata
- Glob_CellId (used as index): study name + sample specific barcode
- Glob_Sample: study name + sample
- Glob_Patient: study name + patient
- Patient: nomification of the original study
- Study: study name
- Tissue: crc or pm or normal
- CellTypeFull: store layered annotation
- CellType: store final annotation
- Batch: Batch information per study

C. Unification of .var metadata
- var_names or index will be ENSG ID for following analysis
- gene_id: replicate of var_names
- gene_name: gene symbol

#### Step one: read in and unify data type and meta data(obs)

In [3]:
### ======
# crc01 data
### ======
CurrentDir = DataRootDir / "crc01"
adata_crc01 = sc.read_h5ad(CurrentDir / "HRA003293.h5ad")

# remove log data and transform raw counts to int32
del adata_crc01.layers["data"]
del adata_crc01.X
adata_crc01.layers["counts"] = ut.to_int32_counts(
    adata_crc01.layers["counts"]
)

In [4]:
# modify obs
crc01_obs = adata_crc01.obs.copy()
obs_list = ['sample','cells','tissue','patient','method','celltype']
crc01_obs = crc01_obs[obs_list]
crc01_obs['Study'] = 'crc01'
rename_dict = {"patient": "Patient","tissue":"Tissue","sample":"Sample","cells":"CellID", "method":"Batch","celltype":"CellType"}
crc01_obs = crc01_obs.rename(columns=rename_dict)
crc01_obs['Glob_CellID'] = crc01_obs['Study'] + '-' + crc01_obs['CellID']
crc01_obs['Glob_Sample'] = crc01_obs['Study'] + '-' + crc01_obs['Sample']
crc01_obs['Glob_Patient'] = crc01_obs['Study'] + '-' + crc01_obs['Patient']
crc01_obs['Batch'] = 'scRNA_' + crc01_obs['Batch']
crc01_obs = crc01_obs.set_index("Glob_CellID", drop=False)
adata_crc01.obs = crc01_obs.copy()
del crc01_obs

In [5]:
# rule out liver metastasis
adata_crc01 = adata_crc01[adata_crc01.obs["Tissue"] != "lm"].copy()

In [6]:
# reset other analysis
adata_crc01.obsm = {}
adata_crc01.obsp = {}
# reset var
adata_crc01.var = pd.DataFrame({'gene_name': adata_crc01.var_names})

/home/zhekai/miniconda3/envs/pm/lib/python3.13/functools.py:934: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


In [7]:
### ======
# crc03 data (scRNA-seq, GSE234804)
### ======
CurrentDir = DataRootDir / "crc03/GSE234804"
FileList = sorted(
    file for file in os.listdir(CurrentDir) if file.endswith(".h5ad")
)
adata_list = []

for file in FileList:
    name = Path(file).stem.split("_")[1]
    adata = sc.read_h5ad(CurrentDir / file)

    # Clean up layers
    for layer_name in list(adata.layers.keys()):
        if layer_name == "scale.data" or layer_name == "data" :
            del adata.layers[layer_name]
            continue
        elif layer_name == "counts":
            adata.layers[layer_name] = sp.sparse.csr_matrix(
                ut.to_int32_counts(adata.layers[layer_name])
            )
            continue
    
    # Build metadata
    adata.obs["Study"] = "crc03"
    adata.obs["Sample"] = name
    adata.obs["Batch"] = "scRNA"
    adata.obs["Glob_CellID"] = "crc03-" + name + "_" + adata.obs.index
    adata.obs["Glob_Sample"] = "crc03-" + name

    # use regex to find prefix and num
    prefix, num = re.findall(r"^(\w+)(\d+)$", name)[0]
    if prefix == "PC":
        tissue = "pm"
    elif prefix == "CRC":
        tissue = "crc"
    patient = "patient-" + num
    
    # assign value 
    adata.obs["Tissue"] = tissue
    adata.obs["Patient"] = patient
    adata.obs["Glob_Patient"] = "crc03-" + adata.obs["Patient"]
    adata.obs = adata.obs.set_index("Glob_CellID", drop=False)

    adata_list.append(adata)

In [8]:
adata_crc03 = ad.concat(
    adata_list,
    join="inner",
    merge="first",
    uns_merge=None,
)
del adata_list
# Reset analysis artifacts, keep var (contains gene_ids)
adata_crc03.obsm = {}
adata_crc03.obsp = {}

In [9]:
### ======
# crc06 data (snRNA-seq, GSE183916)
### ======
adata_crc06 = sc.read_h5ad(DataRootDir / "crc06/GSE183916/GSE183916.h5ad")
# only use raw data
del adata_crc06.layers["data"]
adata_crc06.layers["counts"] = ut.to_int32_counts(adata_crc06.layers["counts"])

In [10]:
# adjusting obs
crc06_obs = adata_crc06.obs.copy()
obs_list = ['loc','pat_id']
crc06_obs = crc06_obs[obs_list]
crc06_obs["Study"] = "crc06"
rename_dict = {"pat_id": "Patient","loc": "Tissue",}
crc06_obs = crc06_obs.rename(columns=rename_dict)
crc06_obs["Tissue"] = crc06_obs["Tissue"].map({"metastasis": "pm","primary": "crc",})
crc06_obs["Sample"] = crc06_obs["Patient"] + "_" + crc06_obs["Tissue"]
crc06_obs["Batch"] = "snRNA"
crc06_obs["Glob_CellID"] = ("crc06-" + crc06_obs.index)
crc06_obs["Glob_Sample"] = "crc06-" + crc06_obs["Sample"]
crc06_obs["Glob_Patient"] = "crc06-" + crc06_obs["Patient"]
# dealing index
crc06_obs['CellID'] = crc06_obs.index
crc06_obs = crc06_obs.set_index("Glob_CellID", drop=False)
adata_crc06.obs = crc06_obs.copy()
del crc06_obs
adata_crc06.var = pd.DataFrame({'gene_name':adata_crc06.var.index})

/home/zhekai/miniconda3/envs/pm/lib/python3.13/functools.py:934: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


In [11]:
### ======
# crc09 data (scRNA)
### ======
# getting raw data
# here, .X stores integer
adata_crc09 = sc.read_h5ad(DataRootDir / "crc09/pelka-2021.h5ad")
adata_crc09.layers['counts']= ut.to_int32_counts(adata_crc09.X.copy())
del adata_crc09.X

/home/zhekai/miniconda3/envs/pm/lib/python3.13/site-packages/anndata/_core/anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [12]:
# getting info from obs
crc09_obs = adata_crc09.obs.copy()
obs_list = ['batchID','clTopLevel','clMidwayPr','cl295v11SubFull','SPECIMEN_TYPE','PROCESSING_TYPE','SINGLECELL_TYPE','PID']
# transform categorical to string
for name in obs_list:
    crc09_obs[name] = crc09_obs[name].astype(str)
crc09_obs = crc09_obs[obs_list]
crc09_obs['CellID'] = crc09_obs.index.astype(str)
rename_dict = {'PID':'Patient', 'batchID':'Sample','SPECIMEN_TYPE':'Tissue','SINGLECELL_TYPE':'Batch','clMidwayPr':'CellType'}
crc09_obs = crc09_obs.rename(columns = rename_dict)
crc09_obs['Tissue'] = crc09_obs['Tissue'].map({'N':'normal','T':'crc'})
# generate new cols
crc09_obs['Study'] = 'crc09'
crc09_obs['Glob_Sample'] = 'crc09-' + crc09_obs['Sample']
crc09_obs['Glob_Patient'] = 'crc09-' + crc09_obs['Patient']
crc09_obs['Glob_CellID'] = 'crc09-' + crc09_obs.index.astype(str)
crc09_obs = crc09_obs.set_index("Glob_CellID", drop=False)
crc09_obs['CellTypeFull'] = crc09_obs['clTopLevel'] + '_' + crc09_obs['CellType'] + '_' + crc09_obs['cl295v11SubFull']
adata_crc09.obs = crc09_obs
del crc09_obs
# only unsorted data is wanted, as we want to analysis cell proportion
adata_crc09 = adata_crc09[adata_crc09.obs['PROCESSING_TYPE'] == 'unsorted'].copy()
adata_crc09.obs = adata_crc09.obs.drop(columns = ['clTopLevel','cl295v11SubFull','PROCESSING_TYPE'])

/home/zhekai/miniconda3/envs/pm/lib/python3.13/site-packages/anndata/_core/anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [13]:
# dealing var and index
adata_crc09.var["gene_ids"].is_unique
adata_crc09.var['gene_name'] = adata_crc09.var.index
adata_crc09.var = adata_crc09.var.rename(columns= {"gene_ids":"gene_id"})
# remove ADT(mainly membrane molecules involving in immunology)
adata_crc09 = adata_crc09[:, adata_crc09.var["gene_id"].str.startswith("ENSG")].copy()
# remove version suffix
# .str is the pd.Serie accessor to iterate through all elements
adata_crc09.var["gene_id"] = (adata_crc09.var["gene_id"].str.split(".").str[0])

/home/zhekai/miniconda3/envs/pm/lib/python3.13/site-packages/anndata/_core/anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [14]:
adata_crc09.var["gene_id"].is_unique
len(adata_crc09.var_names[adata_crc09.var_names.duplicated()])
#####
# since ENSG ids are duplicated (lucky enough only 28 of them))
# we need to add them up. The following is written by ChatGPT
#####
# map duplicated gene_id -> one column
codes, unique_gene_ids = pd.factorize(adata_crc09.var["gene_id"],sort=False)

mapping = sp.sparse.csr_matrix(
    (
        np.ones(adata_crc09.n_vars, dtype=np.int8),
        (np.arange(adata_crc09.n_vars), codes)
    ),
    shape=(adata_crc09.n_vars, len(unique_gene_ids))
)

# sum counts of duplicated genes
counts = adata_crc09.layers["counts"] @ mapping

# construct new var
var = (
    adata_crc09.var
    .groupby("gene_id", sort=False)
    .first()
    .loc[unique_gene_ids]
)

var["gene_id"] = var.index
var.index.name = None

# rebuild AnnData
adata_crc09 = ad.AnnData(
    X=None,
    obs=adata_crc09.obs.copy(),
    var=var,
)

adata_crc09.layers["counts"] = counts.astype(np.int32)

#####
# End of AI generation content
#####

adata_crc09.var = adata_crc09.var.set_index('gene_id',drop=False)
adata_crc09.var.index.name = None
adata_crc09.var = adata_crc09.var.drop(columns = ['genome','feature_types'])

In [15]:
# verify no duplicated vars
adata_crc09.var["gene_id"].is_unique
len(adata_crc09.var_names[adata_crc09.var_names.duplicated()])

0

In [16]:
### ======
# crc10 data (scRNA-seq, SMC+KUL3)
### ======
# read in data
adata_crc10_KUL3 = sc.read_h5ad(DataRootDir / "crc10/lee-2020-KUL3.h5ad")
adata_crc10_SMC = sc.read_h5ad(DataRootDir / "crc10/lee-2020-SMC.h5ad")

for adata in [adata_crc10_KUL3,adata_crc10_SMC]:
    adata.layers['counts']= ut.to_int32_counts(adata.X.copy())
    del adata.X
del adata

In [17]:
# KUL3 obs
KUL3_obs = adata_crc10_KUL3.obs.copy()
rename_dict = {'Cell_type':'CellType','Class':'Tissue'}
KUL3_obs = KUL3_obs.rename(columns = rename_dict)
# transform categorical to string
for name in list(KUL3_obs.columns):
    KUL3_obs[name] = KUL3_obs[name].astype(str)
# generate new cols
KUL3_obs['Study'] = 'crc10-KUL3'
KUL3_obs['Glob_Sample'] = 'crc10-' + KUL3_obs['Sample']
KUL3_obs['Glob_Patient'] = 'crc10-' + KUL3_obs['Patient']
KUL3_obs['Glob_CellID'] = 'crc10-' + KUL3_obs.index.astype(str)
KUL3_obs['CellID'] = KUL3_obs.index
KUL3_obs['CellTypeFull'] = KUL3_obs['CellType'] + '_' + KUL3_obs['Cell_subtype']
KUL3_obs['Batch'] = 'scRNA'
KUL3_obs = KUL3_obs.set_index("Glob_CellID", drop=False)
KUL3_obs = KUL3_obs.drop(columns = ['study','Cell_subtype'])
KUL3_obs['Tissue'] = KUL3_obs['Tissue'].map({'Tumor':'crc','Normal':'normal','Border':'crc'})
adata_crc10_KUL3.obs = KUL3_obs
# SMC obs
SMC_obs = adata_crc10_SMC.obs.copy()
rename_dict = {'Cell_type':'CellType','Class':'Tissue'}
SMC_obs = SMC_obs.rename(columns = rename_dict)
# transform categorical to string
for name in list(SMC_obs.columns):
    SMC_obs[name] = SMC_obs[name].astype(str)
# generate new cols
SMC_obs['Study'] = 'crc10-SMC'
SMC_obs['Glob_Sample'] = 'crc10-' + SMC_obs['Sample']
SMC_obs['Glob_Patient'] = 'crc10-' + SMC_obs['Patient']
SMC_obs['Glob_CellID'] = 'crc10-' + SMC_obs.index.astype(str)
SMC_obs['CellID'] = SMC_obs.index
SMC_obs['CellTypeFull'] = SMC_obs['CellType'] + '_' + SMC_obs['Cell_subtype']
SMC_obs['Batch'] = 'scRNA'
SMC_obs = SMC_obs.set_index("Glob_CellID", drop=False)
SMC_obs = SMC_obs.drop(columns = ['Cell_subtype'])
SMC_obs['Tissue'] = SMC_obs['Tissue'].map({'Tumor':'crc','Normal':'normal','Border':'crc'})
adata_crc10_SMC.obs = SMC_obs

In [18]:
# concat
adata_crc10 = ad.concat([adata_crc10_KUL3,adata_crc10_SMC],
                        join="outer",merge="first",uns_merge=None,)
# var
adata_crc10.var = pd.DataFrame({'gene_name': adata_crc10.var_names})

/home/zhekai/miniconda3/envs/pm/lib/python3.13/functools.py:934: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


#### Step two: unify var data

In [19]:
# default version: GRCh38.111
# for those who have designated version, use their own

In [20]:
### GTF gene name -> ENSGID
MappingPath = Path.home() / "project/crc-pm/GTF"
Mapping = {}
for ver in os.listdir(MappingPath):
    Mapping[ver] = pd.read_csv(MappingPath / ver /'gene_id_gene_name_gene_biotype.csv', dtype=str)
# multiple gene_ids may map to the same gene_name
# Mapping["GRCh38.111"]["gene_name"].duplicated().sum()

# in order to keep most of the information, only keep the ENSG with meaningful biotype of the same gene_name
# setting priority; for those who share same priority, leave it to fate to decide
priority = {
    "protein_coding": 0,
    "lncRNA": 1,
    "miRNA": 2,"snRNA": 2,"snoRNA": 2,"scaRNA": 2,"scRNA": 2,"ribozyme": 2,"vault_RNA": 2,
    "IG_V_gene": 3,"IG_D_gene": 3,"IG_J_gene": 3,"IG_C_gene": 3,"TR_V_gene": 3,"TR_D_gene": 3,"TR_J_gene": 3,"TR_C_gene": 3,
    "rRNA": 4,"Mt_rRNA": 4,"Mt_tRNA": 4,
    "translated_processed_pseudogene": 5,
    "transcribed_unprocessed_pseudogene": 6,"transcribed_processed_pseudogene": 6,"transcribed_unitary_pseudogene": 6,
    "unprocessed_pseudogene": 7,"processed_pseudogene": 7,"unitary_pseudogene": 7,"rRNA_pseudogene": 7,"IG_V_pseudogene": 7,"IG_C_pseudogene": 7,"IG_J_pseudogene": 7,"TR_V_pseudogene": 7,"TR_J_pseudogene": 7,
    "TEC":8, 'misc_RNA': 8,
}

Mapping_dedup = {}
gene2id_dict = {}
gene2biotype_dict = {}
for ver in Mapping.keys():
    Mapping[ver]["biotype_priority"] = Mapping[ver]["gene_biotype"].map(priority).fillna(9)
    Mapping[ver] = Mapping[ver].sort_values("biotype_priority")
    Mapping_dedup[ver] = Mapping[ver].drop_duplicates("gene_name", keep="first")
    gene2id_dict[ver] = dict(zip(Mapping_dedup[ver]["gene_name"], Mapping_dedup[ver]["gene_id"]))
    gene2biotype_dict[ver] = dict(zip(Mapping_dedup[ver]["gene_name"], Mapping_dedup[ver]["gene_biotype"]))

In [21]:
### =====
# adata_crc01
### =====
# map
adata_crc01.var['gene_id'] = adata_crc01.var['gene_name'].map(gene2id_dict['GRCh38.111'])
adata_crc01.var['gene_biotype'] = adata_crc01.var['gene_name'].map(gene2biotype_dict['GRCh38.111'])
# those NaN cannot be mapped even stripped of suffix .x 
adata_crc01 = adata_crc01[:, ~adata_crc01.var["gene_id"].isna()].copy()
adata_crc01.var = adata_crc01.var.set_index('gene_id', drop = False)
adata_crc01.var.index.name = None

In [22]:
### =====
# adata_crc03
### =====
adata_crc03.var['gene_name'] = adata_crc03.var.index
adata_crc03.var = adata_crc03.var.rename(columns = {'gene_ids':'gene_id'})
adata_crc03.var = adata_crc03.var.set_index('gene_id', drop = False)
adata_crc03.var['gene_biotype'] = adata_crc03.var['gene_name'].map(gene2biotype_dict['GRCh38.98'])
adata_crc03.var['gene_biotype'].isna().sum()

np.int64(296)

In [23]:
### =====
# adata_crc06
### =====
adata_crc06.var['gene_id'] = adata_crc06.var['gene_name'].map(gene2id_dict['GRCh38.98'])
adata_crc06.var['gene_biotype'] = adata_crc06.var['gene_name'].map(gene2biotype_dict['GRCh38.98'])
adata_crc06.var = adata_crc06.var.set_index('gene_id',drop = False)
# 664 gene cannot be mapped, which is acceptible
ids = 0
for _ in list(adata_crc06.var['gene_id'].isna()):   
    if _: ids += 1
print(ids)

adata_crc06 = adata_crc06[:,adata_crc06.var["gene_id"].notna()].copy()

664


In [24]:
### =====
# adata_crc09
### =====
adata_crc09.var['gene_biotype'] = adata_crc09.var['gene_name'].map(gene2biotype_dict['GRCh38.84'])
# 3779 not matched,matched 91%
ids = 0
for _ in list(adata_crc09.var['gene_biotype'].isna()):   
    if _: ids += 1
print(100*(1-ids/len(adata_crc09.var['gene_biotype'])))

91.22183507549362


In [25]:
### =====
# adata_crc10
### =====
adata_crc10.var['gene_id'] = adata_crc10.var['gene_name'].map(gene2id_dict['GRCh38.84'])
adata_crc10.var['gene_biotype'] = adata_crc10.var['gene_name'].map(gene2biotype_dict['GRCh38.84'])
adata_crc10.var = adata_crc10.var.set_index('gene_id',drop = False)
print(adata_crc10.var.shape)
ids = 0
for _ in list(adata_crc10.var['gene_id'].isna()):   
    if _: ids += 1
print(ids)
ids = 0
for _ in list(adata_crc10.var['gene_biotype'].isna()):   
    if _: ids += 1
print(ids)
adata_crc10 = adata_crc10[:,adata_crc10.var["gene_id"].notna()].copy()

(33694, 3)
34
34


##### current practise:
crc01: GRCh38.111; crc03:GRCh38.98; crc06: GRCh38.98; crc09: GRCh38.84; crc10: GRCh38.84

In [26]:
from collections import Counter
from math import floor

adatalist = [adata_crc01,adata_crc03,adata_crc06,adata_crc09,adata_crc10]
thres=0.75
# calculate thres
min_studies = floor(len(adatalist) * thres)

gene_counter = Counter()
for adata in adatalist:
    if not adata.var_names.is_unique:
        raise ValueError("adata.var_names must be unique.")
    gene_counter.update(adata.var_names)
keep_genes = {gene for gene, count in gene_counter.items() if count >= min_studies}

In [27]:
# concatenate
concat_list = [adata[:, adata.var_names.isin(keep_genes)].copy() for adata in adatalist]
def merge_var(x):
    if isinstance(x[0], pd.DataFrame):
        return pd.concat(x).groupby(level=0, sort=False).first()
    return {}
concat_adata = ad.concat(
    concat_list, axis="obs", join="outer", merge=merge_var, fill_value=0,
)
print(concat_adata.var.isna().sum())

gene_name       0
gene_id         0
gene_biotype    0
dtype: int64


### Data Export

In [28]:
export_dir = Path("../../result/curated/merged.h5ad")
concat_adata.write_h5ad(export_dir)